# 조건 A/B 비교 보고서 템플릿

저장된 Run의 조건·지표·질문·답변·근거·판정을 기계적으로 표시합니다.
해석·결론을 새로 작성하거나 생성하지 않습니다. 기본 입력은 실제 MLflow Run입니다.
템플릿을 `../reports/`로 복사해 입력 Run과 표시 범위를 설정하세요.
HTML에는 `report_view`가 포함되며 줌·hover·표 검색·정렬은 export 후에도 동작합니다.

In [ ]:
import json
import os

import hvplot.pandas  # noqa: F401 — pandas에 hvPlot 연결
import mlflow
import pandas as pd
import panel as pn
from IPython.display import display

from elice_reporting import (
    case_view,
    comparison_frames,
    demo,
    export_html,
    find_root,
    from_files,
    from_mlflow,
    interactive_table,
    metric_card,
    report_purpose_callout,
    text_block,
)

pn.extension()
ROOT = find_root()

In [ ]:
# 보고서마다 수정하는 입력입니다. 인증정보는 코드에 쓰지 않습니다.
SOURCE_MODE = "mlflow"  # demo | mlflow | local
RUN_ID = ""  # analysis 또는 comparison run ID
REPORT_TITLE = "조건 A/B 비교 보고서"
OUTPUT_HTML = ROOT / "artifacts/reports/comparison-draft.html"
LOCAL_REPORT = ROOT / "artifacts/evaluation/comparison-result.json"
LOCAL_INPUTS = ROOT / "artifacts/evaluation/comparison-inputs.json"

# 미지정 시 원본 순서의 첫 대상을 사용합니다. 최고 점수를 자동 선택하지 않습니다.
FOCUS_RATER = None
CASE_IDS = []  # 비워 두면 저장 순서대로 전체 문항을 표시합니다.
FOCUS_METRIC = "rubric.correctness.mean"

In [ ]:
if SOURCE_MODE == "demo":
    data = demo("comparison")
elif SOURCE_MODE == "mlflow":
    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    client = mlflow.MlflowClient()
    if not RUN_ID:
        experiment = client.get_experiment_by_name(
            os.environ.get("QA_MLFLOW_EXPERIMENT_NAME", "elice-techblog-rag-evaluation")
        )
        if experiment is None:
            raise ValueError("실험을 찾을 수 없습니다. 환경 설정을 확인하세요.")
        runs = client.search_runs(
            [experiment.experiment_id],
            filter_string=(
                "tags.`evaluation.kind` = 'comparison' AND attributes.status = 'FINISHED'"
            ),
            max_results=20,
        )
        display(
            pd.DataFrame(
                [{"run_id": r.info.run_id, "name": r.data.tags.get("mlflow.runName")} for r in runs]
            )
        )
        raise ValueError("위 목록에서 RUN_ID를 설정하고 다시 실행하세요.")
    data = from_mlflow("comparison", RUN_ID, ROOT / "artifacts/reports/cache")
else:
    if SOURCE_MODE != "local":
        raise ValueError("SOURCE_MODE는 demo, mlflow, local 중 하나여야 합니다.")
    data = from_files("comparison", LOCAL_REPORT, LOCAL_INPUTS)

# SDK로 metric/param을 추가 조회할 수도 있습니다(읽기 전용).
# run = client.get_run(RUN_ID)
# run.data.metrics

In [ ]:
from elice_reporting.presentation import metric_label

frames = comparison_frames(data)
IS_DEMO = data.source == "demo" or data.report.get("demo", False)
banner = pn.pane.Markdown(
    "## 데모 데이터 — 실제 실험 결과가 아닙니다." if IS_DEMO else "",
    styles={"color": "#a44a08"},
)

DIMENSIONS = {
    "retrieved_context_precision": "누적 검색 정밀도",
    "retrieved_context_recall": "누적 검색 재현율",
    "generation_context_precision": "최종 문맥 정밀도",
    "generation_context_recall": "최종 문맥 재현율",
    "correctness": metric_label("correctness"),
    "completeness": metric_label("completeness"),
    "groundedness": metric_label("groundedness"),
    "relevance": metric_label("relevance"),
}


def number(value, *, percent=False, signed=False):
    if value is None or pd.isna(value):
        return "계산 불가"
    return format(
        value, "+.1%" if signed and percent else ".1%" if percent else "+.3f" if signed else ".3f"
    )

## 1. 저장된 조건과 요약 지표

표시할 평가자와 지표는 설정값을 따릅니다. 미지정 평가자는 원본 순서의 첫 대상입니다.
집계값·미정의 사유·유효 표본은 원본 artifact를 사용합니다.

In [ ]:
quality = frames["quality"]
raw_scores = quality[quality["metric"].str.startswith("rubric.")].copy()
available_raters = raw_scores["rater"].drop_duplicates().tolist()
selected_rater = FOCUS_RATER or (available_raters[0] if available_raters else None)
selected = raw_scores[raw_scores["rater"] == selected_rater]
focus_rows = selected[selected["metric"] == FOCUS_METRIC]
focus = focus_rows.iloc[0] if not focus_rows.empty else None
duration_rows = frames["resources"][frames["resources"]["metric"] == "qa_duration_ms"]
duration = duration_rows.iloc[0] if not duration_rows.empty else None
purpose = data.report["purpose"]
conditions_view = (
    pn.Column(
        *[
            text_block(row["path"], f"A: {row['baseline']} → B: {row['candidate']}")
            for _, row in frames["conditions"].iterrows()
        ],
        sizing_mode="stretch_width",
    )
    if not frames["conditions"].empty
    else pn.pane.Markdown("QA 설정 변경 없음. Judge 변경은 상세 정보를 확인하세요.")
)
summary_cards = pn.FlexBox(
    metric_card(
        "공통 문항",
        str(len(data.report["coverage"]["shared_record_ids"])),
        "개별 지표의 유효 문항 수와는 다름",
    ),
    metric_card(
        "주 지표 · A → B",
        f"{number(focus['baseline_mean'])} → {number(focus['candidate_mean'])}"
        if focus is not None
        else "기록 없음",
        f"{selected_rater} / {FOCUS_METRIC}",
    ),
    metric_card(
        "주 지표 변화 · B−A",
        number(focus["delta_mean"], signed=True) if focus is not None else "계산 불가",
        f"대응 {focus['paired_cases']}문항 · 제외 {focus['excluded_cases']}문항"
        if focus is not None
        else "선택 지표 없음",
    ),
    metric_card(
        "평균 응답 시간 · A → B",
        f"{number(duration['baseline_mean'])} → {number(duration['candidate_mean'])} ms"
        if duration is not None
        else "기록 없음",
        "qa_duration_ms · ms",
    ),
    sizing_mode="stretch_width",
)
interval_note = (
    f"주 지표의 평균 차이 구간: {focus['ci_low']} ~ {focus['ci_high']}. "
    f"계산 불가 사유: {focus.get('interval_unavailable_reason') or '없음'}."
    if focus is not None
    else "주 지표를 확인할 수 없습니다. FOCUS_METRIC과 FOCUS_RATER를 확인하세요."
)
# Judge 변경 실험은 점수 상승 대신 사람과의 정합성을 우선 확인합니다.
reliability_pairs = [
    row
    for row in data.report["reliability"]["pairs"]
    if row["purpose"] == "human_alignment" and selected_rater in row["raters"]
]
reliability_focus = next(
    (row for row in reliability_pairs if row["dimension"] == "correctness"), None
)
if purpose == "judge":
    summary_cards = pn.FlexBox(
        metric_card(
            "공통 답변 수",
            str(len(data.report["coverage"]["shared_record_ids"])),
            "답변 고정 · Judge 변경",
        ),
        metric_card(
            "사람과의 선형 가중 Cohen κ · A → B",
            (
                f"{number(reliability_focus['baseline']['cohen_kappa'])} → "
                f"{number(reliability_focus['candidate']['cohen_kappa'])}"
            )
            if reliability_focus
            else "계산 불가",
            "정확성 / 선택 모델과 사람의 판정",
        ),
        metric_card(
            "κ 변화 · B−A",
            number(reliability_focus["delta"]["cohen_kappa"], signed=True)
            if reliability_focus
            else "계산 불가",
            "품질 점수 상승과 구분",
        ),
        sizing_mode="stretch_width",
    )
    interval_note = (
        "품질 점수의 구간은 κ의 구간이 아닙니다. 이 보고서는 κ의 구간을 계산하지 않습니다."
    )

## 2. 주요 그래프

공통 rubric 등급 A/B를 나란히 보여줍니다. 동일한 평가자의 점수이며, 평가자 변경은 목적에 따라 별도로 해석합니다.
시간·토큰 등 단위가 다른 수치는 품질 그래프에 섞지 않습니다.

In [ ]:
chart_data = selected.dropna(subset=["baseline_mean", "candidate_mean"]).copy()
chart_data["평가 항목"] = chart_data["metric"].str.removeprefix("rubric.").str.removesuffix(".mean")
chart_data["평가 항목"] = chart_data["평가 항목"].map(DIMENSIONS).fillna(chart_data["평가 항목"])
chart_data = chart_data.melt(
    id_vars=["평가 항목"],
    value_vars=["baseline_mean", "candidate_mean"],
    var_name="조건",
    value_name="평균 점수",
)
chart_data["조건"] = chart_data["조건"].map(
    {"baseline_mean": "A · 기준", "candidate_mean": "B · 변경"}
)
quality_view = (
    chart_data.hvplot.bar(
        x="평가 항목",
        y="평균 점수",
        by="조건",
        ylim=(0, max(data.inputs["baseline"]["gradings"][0]["rubric"]["scale"]) + 0.05),
        height=370,
        responsive=True,
        rot=25,
        legend="bottom",
        cmap=["#95a4b5", "#2563a6"],
        tools=["hover"],
    )
    if not chart_data.empty
    else pn.pane.Markdown("대응 가능한 rubric 등급가 없습니다.")
)
if purpose == "judge":
    reliability_chart = pd.DataFrame(
        [
            {
                "평가 항목": DIMENSIONS.get(row["dimension"], row["dimension"]),
                "조건": label,
                "선형 가중 Cohen κ": row[side]["cohen_kappa"],
            }
            for row in reliability_pairs
            for side, label in (("baseline", "A · 기준 Judge"), ("candidate", "B · 변경 Judge"))
        ],
        columns=["평가 항목", "조건", "선형 가중 Cohen κ"],
    ).dropna(subset=["선형 가중 Cohen κ"])
    quality_view = (
        reliability_chart.hvplot.bar(
            x="평가 항목",
            y="선형 가중 Cohen κ",
            by="조건",
            ylim=(-1, 1),
            height=350,
            responsive=True,
            rot=0,
            legend="bottom",
            cmap=["#95a4b5", "#2563a6"],
            tools=["hover"],
        )
        if not reliability_chart.empty
        else pn.pane.Markdown("대응하는 사람–모델 κ가 없습니다.")
    )

## 3. 문항별 저장 결과

기본값은 전체 문항을 저장 순서대로 표시합니다. CASE_IDS를 지정하면 해당 ID만 표시하고,
보고서에 설정값과 적용 순서·표시 수를 명시합니다. 판정·점수에 따른 사례 선별은 하지 않습니다.

In [ ]:
transitions = frames["transitions"]
focal_cases = transitions[
    (transitions["metric"] == FOCUS_METRIC) & (transitions["rater"] == selected_rater)
]
available_ids = data.report["coverage"]["shared_record_ids"]
case_ids = list(dict.fromkeys(CASE_IDS)) if CASE_IDS else list(available_ids)
if not set(case_ids) <= set(available_ids):
    raise ValueError("CASE_IDS에 공통 문항에 없는 ID가 있습니다.")
case_scope = (
    f"{'CASE_IDS 지정 순서' if CASE_IDS else '전체 공통 문항 · 저장 순서'} / "
    f"{len(case_ids)} / {len(available_ids)}문항\n표시 ID: {case_ids}"
)
examples = []
for record_id in case_ids:
    rows = focal_cases[focal_cases["record_id"] == record_id]
    outcome = rows.iloc[0] if not rows.empty else None
    examples.append(
        pn.Card(
            text_block(
                "저장된 주 지표 변화",
                f"{outcome['change']} · A {outcome['baseline']} → B {outcome['candidate']}"
                if outcome is not None
                else "대응 점수 없음",
            ),
            case_view(data.inputs["baseline"]["bundle"], record_id, title="A · 기준 답변"),
            case_view(data.inputs["candidate"]["bundle"], record_id, title="B · 변경 답변"),
            title=f"문항 {record_id}",
            collapsed=True,
            sizing_mode="stretch_width",
        )
    )
examples_view = pn.Column(*examples, sizing_mode="stretch_width")

## 4. 보고서 조합

저장된 조건·지표·문항별 결과와 전체 상세 표를 조합합니다. 해석·결론 입력란은 없습니다.

In [ ]:
details = pn.Accordion(
    ("평가자별 전체 품질 · 평균 차이 구간 · 제외 수", interactive_table(quality)),
    ("응답 시간·토큰·호출 수", interactive_table(frames["resources"])),
    ("사람–모델 및 모델 간 정합성 변화", interactive_table(frames["pairs"])),
    ("전체 문항별 변화·제외 사유", interactive_table(transitions)),
    ("A 전체 질문·답변", interactive_table(frames["baseline_cases"])),
    ("B 전체 질문·답변", interactive_table(frames["candidate_cases"])),
    (
        "전체 조건 변경과 재현 정보",
        pn.Column(
            interactive_table(frames["conditions"]),
            text_block(
                "프로토콜·문항 범위·Judge 프로필 변경",
                json.dumps(
                    {
                        "baseline": data.report["baseline"],
                        "candidate": data.report["candidate"],
                        "protocol": data.report["protocol"],
                        "coverage": data.report["coverage"],
                        "grading_profile_changes": data.report["grading_profile_changes"],
                    },
                    ensure_ascii=False,
                    indent=2,
                ),
            ),
        ),
    ),
    active=[],
    sizing_mode="stretch_width",
)
report_view = pn.Column(
    pn.pane.Markdown("# " + REPORT_TITLE),
    report_purpose_callout("comparison"),
    banner,
    pn.pane.Markdown("## 1. 비교 조건"),
    text_block("저장된 비교 가설", data.report["hypothesis"]),
    text_block("비교 유형과 주 평가자", f"{purpose} / {selected_rater}"),
    conditions_view,
    pn.pane.Markdown("## 2. 저장된 요약 지표"),
    summary_cards,
    text_block("구간과 표본", interval_note),
    pn.pane.Markdown(
        "## 3. 항목별 A/B 점수" if purpose == "system" else "## 3. 항목별 A/B 선형 가중 Cohen κ"
    ),
    text_block("그래프의 평가자", selected_rater),
    quality_view,
    pn.pane.Markdown("## 4. 문항별 저장 결과"),
    text_block("표시 범위", case_scope),
    examples_view,
    pn.pane.Markdown("## 5. 상세 데이터와 재현 정보"),
    details,
    sizing_mode="stretch_width",
    max_width=1100,
    styles={"margin": "0 auto"},
)
report_view

## 독립 HTML export

`report_view`에 포함한 조건·차트·표·문항별 기록을 로컬 HTML로 저장합니다.
업로드는 수행하지 않으며 MLflow 첨부 예제는 주석 상태입니다.

In [ ]:
html_path = export_html(report_view, OUTPUT_HTML, title=REPORT_TITLE, overwrite=True)
html_path

# MLflow에 첨부하려면 결과와 대상 run을 확인한 뒤 직접 실행하세요.
# client.log_artifact(RUN_ID, str(html_path), artifact_path="reports")